In [1]:
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import time
from annoy import AnnoyIndex

def detect_faces_dnn(image):
    net = cv2.dnn.readNetFromCaffe("ssd/deploy.prototxt.txt", "ssd/res10_300x300_ssd_iter_140000.caffemodel")
    h, w = image.shape[:2]
    # Prepare image for DNN processing
    blob = cv2.dnn.blobFromImage(cv2.resize(image, (230, 238)), 1.0, (300, 300), (104.0, 177.0, 123.0))
    net.setInput(blob)
    detections = net.forward()
    faces = []
    for i in range(detections.shape[2]):
        confidence = detections[0, 0, i, 2]
        if confidence > 0.7:  
            box = detections[0, 0, i, 3:7] * [w, h, w, h]
            faces.append(box.astype("int"))
    return faces

np.random.seed(42)

In [4]:
# Load LBPH model, extract features and labels
face_recognizer = cv2.face.LBPHFaceRecognizer.create(radius=1, neighbors=7, grid_x=7, grid_y=7, threshold=60)
face_recognizer.read("models/trained_on_test.yml")
features = np.array(face_recognizer.getHistograms())
size = len(np.array(face_recognizer.getHistograms()))
features = np.reshape(features, (size, -1))
labels = np.array(face_recognizer.getLabels())
labels = labels.ravel()

# Apply PCA on the data and on the test feature
pca = PCA(n_components=100, random_state=42)
features_pca = pca.fit_transform(features)


# Initalize the annoy index. Build and train once, then it can be saved and loaded later.
def build_annoy_index(dataset, path, num_trees=10):
    """
    Build an Annoy index for approximate nearest neighbor search.
    """
    feature_dim = dataset.shape[1]
    # index = AnnoyIndex(feature_dim, metric='euclidean')
    index = AnnoyIndex(feature_dim, metric='manhattan')
    # index = AnnoyIndex(feature_dim, metric='angular')
    # index = AnnoyIndex(feature_dim, metric='dot')
    # index = AnnoyIndex(feature_dim, metric='hamming')
    for i, vector in enumerate(dataset):
        index.add_item(i, vector)
    index.build(num_trees)
    index.save(path)
    return index

def load_annoy_index(dataset, path):
    annoy = AnnoyIndex(dataset.shape[1], metric='manhattan')
    annoy.load(path)
    return annoy


def lsh_search(query, index, dataset_labels, k=1):
    """
    Perform LSH search using Annoy.
    """
    indices, error = index.get_nns_by_vector(query, k, include_distances=True)
    return dataset_labels[indices[0]], error

annoy_index_pca = build_annoy_index(features_pca, path="models/annoy_index_pca_100")

In [5]:
# Run this cell for testing
annoy_index_pca = load_annoy_index(features_pca, path="models/annoy_index_pca_100")

###
# Load features of the test image
face_recognizer2 = cv2.face.LBPHFaceRecognizer.create(radius=1, neighbors=7, grid_x=7, grid_y=7, threshold=60)
# test_img = cv2.imread('test_img2.jpg')
test_img = cv2.imread('test_data/20/duongtest_1.jpg')
gray_img = cv2.cvtColor(test_img, cv2.COLOR_BGR2GRAY)

face = detect_faces_dnn(test_img)
x_start, y_start, x_end, y_end = face[0]
off_set = -10
y_start = y_start + int((y_end-y_start)*0.269) - off_set
roi_gray = gray_img[y_start:y_end, x_start:x_end]
roi_gray = cv2.resize(roi_gray, (300, 300))
face_recognizer2.train(np.array([roi_gray]), np.array(2))
test_instance = face_recognizer2.getHistograms()[0][0]
test_feature = pca.transform([np.array(test_instance)])
###


predicted_label, error = lsh_search(test_feature[0], annoy_index_pca, labels)
predicted_label, error

(1, [6.046245098114014])